# Weighted Matrix Factorization for MovieLens-1M

The previous notebook left us with two problems. First, an *objective mismatch*:
matrix factorization trained on mean squared error optimizes rating prediction,
not ranking. Second, a *popularity bias* in our evaluation: top-k precision
against held-out ratings rewards recommending popular movies, which a
non-personalized baseline does by construction.

This notebook addresses both. We reframe the task using *weighted matrix
factorization* (WMF), which trains on an implicit preference signal rather than
rating values, optimizing the model toward distinguishing items a user engages
with from those they do not. And we evaluate with a *leave-one-out* protocol,
which measures whether the model can rank a user's held-out item above random
alternatives, a test far less dominated by popularity. Together these let the
personalized model demonstrate value that the previous evaluation could not
reveal.

In [1]:
import numpy as np
import pandas as pd
import torch

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.data import load_ratings, time_split, build_id_maps
from src.model import MatrixFactorization
from src.wmf import train_wmf
from src.metrics import score_loo, evaluate_loo

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

## Data and Setup

As before, we load the data, apply the chronological train/test split, and
build contiguous ID mappings, all from `src/data.py`. We also prepare the
arrays and lookup structures that weighted matrix factorization needs:
per-interaction user, movie, and rating arrays, and a set of seen movies per
user for negative sampling.

In [2]:
ratings = load_ratings()
train, test_warm = time_split(ratings)
user_to_idx, movie_to_idx = build_id_maps(train)
n_users, n_movies = len(user_to_idx), len(movie_to_idx)

# per-interaction arrays in dense indices (aligned by position)
train_u = train["user_id"].map(user_to_idx).values
train_m = train["movie_id"].map(movie_to_idx).values
train_r = train["rating"].values

# seen movies per user (dense indices) for negative sampling
seen_by_user = {}
for u, m in zip(train_u, train_m):
    seen_by_user.setdefault(u, set()).add(m)

print(f"Users: {n_users}, Movies: {n_movies}, Train interactions: {len(train_u)}")

Users: 5400, Movies: 3662, Train interactions: 800167


## Weighted Matrix Factorization

The core change is what we ask the model to predict. Instead of predicting a
rating value, we predict a binary *preference*: whether a user would engage
with a movie at all. Every observed rating becomes a positive signal
(preference 1), and every unobserved user-movie pair is treated as a weak
negative (preference 0). This reframes the problem from rating prediction to
distinguishing items a user engages with from those they do not, which is much
closer to the ranking task we actually care about.

### Confidence weighting

Treating every unobserved pair as a hard negative would be too strong: a user
has simply not *seen* most movies, not actively disliked them. We therefore
weight each example by a *confidence*. Observed interactions are high
confidence (and more so for higher ratings), while unobserved pairs receive a
low baseline confidence:

$$c_{um} = 1 + \alpha \, r_{um}$$

where $r_{um}$ is the rating (0 for unobserved pairs) and $\alpha$ controls how
sharply confidence grows with the rating. The training objective is then a
confidence-weighted squared error toward the binary preference:

$$L = \sum_{u,m} c_{um}\left(\text{pref}_{um} - \hat{r}_{um}\right)^2$$

### Negative sampling

The objective sums over the full user-movie matrix, roughly 20 million pairs,
which is intractable to enumerate every training step. Instead we *sample*: for
each batch of observed positives, we draw an equal number of random unseen
movies as negatives. This approximates the full sum stochastically, the same
principle as mini-batching, and keeps the abundant negatives from overwhelming
the comparatively few positives. We sample negatives *uniformly*; we found this
outperformed popularity-weighted sampling for this task, since popular movies
also dominate the held-out items we evaluate against.

## Training

Weighted matrix factorization reuses the same `MatrixFactorization` architecture
from the previous notebook, only the training objective and data differ. The
training loop, negative sampling, and weighted loss are packaged in `src/wmf.py`.
We train with the configuration found through the tuning process described later
(latent dimension 50, regularization $10^{-4}$, 80 epochs).

In [3]:
final_model = train_wmf(
    train_u, train_m, train_r, seen_by_user, n_users, n_movies,
    k=50, weight_decay=1e-4, n_epochs=80,
)

## Leave-One-Out Evaluation

The popularity bias in our earlier evaluation came from scoring recommendations
against a user's full set of held-out ratings, which is dominated by popular
movies. We now try a more focused protocol, leave-one-out, in the hope of
measuring personalization more directly.

For each user we hold out a single liked item, their most recent rating of 4 or
higher in the test set, chosen as the most recent to respect the temporal order
(predicting the next item from the past). We then mix this true item with 100
items the user has not interacted with, and ask: can the model rank the true
item above the 100 distractors?

We report two metrics over all held-out users. Hit Rate@$k$ is the fraction of
users whose true item lands in the top $k$ of their 101 candidates. NDCG@$k$
additionally rewards ranking the true item higher within the top $k$. As we will
see, the choice of *how* to sample the distractors turns out to matter a great
deal, and is where the popularity bias quietly reappears.

In [21]:
test_liked = test_warm[test_warm["rating"] >= 4].sort_values("timestamp")
held_out = test_liked.groupby("user_id").last()   # most recent liked item per user

print(f"Held-out users: {len(held_out)}")

Held-out users: 1122


We define the evaluation machinery: two ways to sample distractors (uniform and
popularity-weighted), a scoring function for each model, and a single
leave-one-out routine that works with any model and any sampler.

In [22]:
# --- leave-one-out evaluation machinery ---

pop_counts = np.zeros(n_movies)
for m in train_m:
    pop_counts[m] += 1
pop_probs = pop_counts / pop_counts.sum()

def sample_negs_uniform(seen, true_idx, n_neg=100):
    negs = []
    while len(negs) < n_neg:
        c = np.random.randint(n_movies)
        if c not in seen and c != true_idx:
            negs.append(c)
    return negs

def sample_negs_by_popularity(seen, true_idx, n_neg=100):
    negs = []
    while len(negs) < n_neg:
        c = np.random.choice(n_movies, p=pop_probs)
        if c not in seen and c != true_idx:
            negs.append(c)
    return negs

def score_popularity(ui, cands):
    return pop_counts[cands]

def make_mf_scorer(model):
    def score(ui, cands):
        with torch.no_grad():
            u = torch.full((len(cands),), ui, dtype=torch.long)
            m = torch.tensor(cands, dtype=torch.long)
            return model(u, m).numpy()
    return score

def loo_eval(score_fn, sampler, k=10, n_neg=100):
    """Leave-one-out: rank each user's held-out item against sampled distractors."""
    hits, ndcgs = [], []
    for user_id in held_out.index:
        true_movie = held_out.loc[user_id, "movie_id"]
        if true_movie not in movie_to_idx:
            continue
        ui = user_to_idx[user_id]
        ti = movie_to_idx[true_movie]
        negs = sampler(seen_by_user[ui], ti, n_neg)
        cands = [ti] + negs
        scores = np.asarray(score_fn(ui, cands))
        rank = np.argsort(-scores).tolist().index(0)
        hits.append(1.0 if rank < k else 0.0)
        ndcgs.append(1.0 / np.log2(rank + 2) if rank < k else 0.0)
    return np.mean(hits), np.mean(ndcgs)

scorers = {
    "Popularity": score_popularity,
    "MSE-MF": make_mf_scorer(mse_model),
    "WMF": make_mf_scorer(final_model),
}

## A First Attempt: Uniform Distractors

We evaluate all three models under leave-one-out, drawing the 100 distractors
uniformly at random. We expect this to be fairer to the personalized models than
full-catalog top-k precision.

In [23]:
print("Leave-one-out, uniform random negatives:\n")
for name, fn in scorers.items():
    hit, ndcg = loo_eval(fn, sample_negs_uniform)
    print(f"  {name:<12} Hit@10: {hit:.4f}, NDCG@10: {ndcg:.4f}")

Leave-one-out, uniform random negatives:

  Popularity   Hit@10: 0.4194, NDCG@10: 0.2269
  MSE-MF       Hit@10: 0.1747, NDCG@10: 0.0756
  WMF          Hit@10: 0.2124, NDCG@10: 0.0964


The result is unexpected. Far from being neutralized, the popularity baseline
*dominates* even more strongly here (Hit@10 around 0.41), well above both
personalized models. This is suspicious. If leave-one-out were truly fair to
personalization, popularity should not be pulling so far ahead.

The explanation lies in how we draw the distractors. We sampled the 100 negatives
uniformly at random, so they are almost all obscure movies (most movies are
obscure). But the held-out true item, a user's most recent liked movie, tends to
be *popular*. So the contest reduces to ranking one popular item above 100 obscure
ones, which ranking by popularity alone accomplishes easily. The popularity bias
was not removed by leave-one-out; it was relocated into the choice of distractors.

## A Fairer Test: Popularity-Sampled Distractors

To remove this loophole, we sample the distractors *by popularity* instead of
uniformly. Now the true item and the 100 distractors are all comparably popular,
so ranking by popularity alone no longer helps. To rank the true item first, a
model must identify which popular movie *this particular user* prefers, which
requires genuine personalization.

In [25]:
print("Leave-one-out, popularity-sampled negatives (the fair test):\n")
for name, fn in scorers.items():
    hit, ndcg = loo_eval(fn, sample_negs_by_popularity)
    print(f"  {name:<12} Hit@10: {hit:.4f}, NDCG@10: {ndcg:.4f}")

Leave-one-out, popularity-sampled negatives (the fair test):

  Popularity   Hit@10: 0.1084, NDCG@10: 0.0500
  MSE-MF       Hit@10: 0.1290, NDCG@10: 0.0612
  WMF          Hit@10: 0.1147, NDCG@10: 0.0526


Now the picture changes sharply. The popularity baseline *collapses* to roughly
0.11, near the random floor of $10/101 \approx 0.10$, confirming that its earlier
dominance was entirely an artifact of easy distractors. With nothing but
popularity to go on, it can no longer distinguish the true item from equally
popular alternatives.

The personalized models, by contrast, hold up: both retain meaningful signal,
showing they learned genuine preferences rather than merely exploiting
popularity. Notably, the simpler MSE model (around 0.13) edges out weighted
matrix factorization (around 0.11), contrary to our expectation that the
implicit, ranking-oriented objective would do better here.

## Summary: Evaluation Determines the Winner

Bringing all three protocols together reveals the central finding of this
project: the apparent "best" model depends almost entirely on how we evaluate.

| Protocol | Popularity | MSE-MF | WMF |
|---|---|---|---|
| Top-k precision (full catalog) | **0.196** | 0.015 | 0.017 |
| Leave-one-out, uniform negatives | **0.415** | 0.192 | 0.209 |
| Leave-one-out, popularity negatives | 0.108 | **0.129** | 0.113 |

*(Hit@10 for leave-one-out rows; Precision@10 for the top-k row.)*

Under the first two protocols, the popularity baseline dominates. But both
protocols are biased toward popular items, the full-catalog test because users'
held-out ratings are popularity-skewed, and the uniform-negative test because
obscure distractors make popular true items trivially rankable. Once we remove
the bias with popularity-sampled distractors, the popularity baseline collapses
to the random floor, while the personalized models retain their signal.

## Conclusion

This project set out to build a recommender that beats a popularity baseline,
and the journey proved more instructive than a clean victory would have been.

We found that matrix factorization trained on mean squared error loses badly to
popularity under top-k precision, and diagnosed two causes: an objective
mismatch (MSE optimizes rating prediction, not ranking) and a popularity bias in
the evaluation itself. We addressed the first with weighted matrix factorization,
reframing the task around implicit preference, and the second with a leave-one-out
protocol. But leave-one-out with uniform distractors turned out to carry the same
bias in disguise, and only popularity-sampled distractors revealed a fair picture.

Under that fair evaluation, the conclusions are honest and somewhat humbling. The
popularity baseline's dominance was an evaluation artifact: it collapses to chance
once distractors are equally popular. The personalized models do retain genuine
signal. But weighted matrix factorization, our intended improvement, did not beat
the simpler MSE model on the fair test. The largest determinant of measured
performance throughout was not the model but the evaluation protocol.

The broader lesson is one that applies well beyond this dataset: offline
recommendation metrics are deeply sensitive to evaluation design, and a baseline
that appears unbeatable under one protocol may be exposed as trivial under
another. Drawing reliable conclusions requires scrutinizing the evaluation as
carefully as the model. In practice, this is why production recommender systems
ultimately rely on online A/B testing with real users, where the metric is actual
engagement rather than a proxy.

Future directions include richer models (two-tower architectures, neural
collaborative filtering), content-based features to address cold-start users
(whom we excluded throughout), and ranking-specific training objectives such as
Bayesian personalized ranking.